# RailGuard AI â€” Notebook 01: Dataset Analysis & EDA

This notebook performs exploratory data analysis on the **RailGuard AI Railway Defect Dataset** (`RailGuardAI/dataset/railway-defect-dataset/`).

### Objectives:
1. Verify dataset structure across Train, Validation, and Test splits.
2. Inspect dataset YAML configuration (`data.yaml`).
3. Analyze annotation counts and class distributions across all 15 defect classes.
4. Inspect bounding box dimensions and image aspect ratios.
5. Render sample images with ground-truth bounding box overlays.

In [ ]:
import os
import glob
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
import numpy as np

# Define base dataset directory
DATASET_DIR = os.path.abspath("../dataset/railway-defect-dataset")
YAML_PATH = os.path.join(DATASET_DIR, "data.yaml")

print(f"Dataset Directory: {DATASET_DIR}")
print(f"YAML Path: {YAML_PATH}")

Dataset Directory: C:\Users\Ankadala Triveni\OneDrive\Documents\RailGuardAI\dataset\railway-defect-dataset
YAML Path: C:\Users\Ankadala Triveni\OneDrive\Documents\RailGuardAI\dataset\railway-defect-dataset\data.yaml


## 1. Dataset Configuration (`data.yaml`)

In [ ]:
with open(YAML_PATH, "r") as f:
    data_config = yaml.safe_load(f)

class_names = data_config.get("names", [])
num_classes = data_config.get("nc", len(class_names))

print(f"Number of Classes (nc): {num_classes}")
print("Defect Class Names:")
for idx, name in enumerate(class_names):
    print(f"  [{idx:2d}] {name}")

Number of Classes (nc): 15
Defect Class Names:
  [ 0] broken_rail_high
  [ 1] broken_rail_low
  [ 2] broken_rail_medium
  [ 3] corrosion_high
  [ 4] corrosion_low
  [ 5] corrosion_medium
  [ 6] head_checks_high
  [ 7] head_checks_low
  [ 8] head_checks_medium
  [ 9] railway_join_bar_defects_high
  [10] railway_join_bar_defects_low
  [11] railway_join_bar_defects_medium
  [12] squats_high
  [13] squats_low
  [14] squats_medium


## 2. Dataset Split Counts (Train / Val / Test)

In [ ]:
splits = ["train", "val", "test"]
split_summary = []

total_images = 0
total_labels = 0

for s in splits:
    img_dir = os.path.join(DATASET_DIR, "images", s)
    lbl_dir = os.path.join(DATASET_DIR, "labels", s)
    
    img_files = glob.glob(os.path.join(img_dir, "*.jpg"))
    lbl_files = glob.glob(os.path.join(lbl_dir, "*.txt"))
    
    # Count total bounding boxes
    bbox_count = 0
    for lf in lbl_files:
        with open(lf, "r") as f:
            bbox_count += len([line for line in f.readlines() if line.strip()])
            
    split_summary.append({
        "Split": s.capitalize(),
        "Images": len(img_files),
        "Labels": len(lbl_files),
        "Bounding Boxes": bbox_count
    })
    total_images += len(img_files)
    total_labels += bbox_count

df_summary = pd.DataFrame(split_summary)
df_summary["Image %"] = (df_summary["Images"] / total_images * 100).round(2)
print(df_summary.to_string(index=False))
print(f"\nTotal Images: {total_images}")
print(f"Total Bounding Boxes: {total_labels}")

     Split  Images  Labels  Bounding Boxes  Image %
     Train    1626    1626            3831    69.97
Validation     464     464            1040    19.97
      Test     234     234             621    10.07

Total Images: 2324
Total Bounding Boxes: 5492


## 3. Class Distribution Analysis

In [ ]:
class_counts = {name: 0 for name in class_names}

for s in splits:
    lbl_dir = os.path.join(DATASET_DIR, "labels", s)
    for lf in glob.glob(os.path.join(lbl_dir, "*.txt")):
        with open(lf, "r") as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls_id = int(parts[0])
                    if 0 <= cls_id < len(class_names):
                        class_counts[class_names[cls_id]] += 1

df_classes = pd.DataFrame(list(class_counts.items()), columns=["Defect Class", "Annotation Count"])
df_classes = df_classes.sort_values(by="Annotation Count", ascending=False).reset_index(drop=True)
df_classes["Percentage (%)"] = (df_classes["Annotation Count"] / total_labels * 100).round(2)

print(df_classes.to_string(index=False))

                   Defect Class  Annotation Count  Percentage (%)
                head_checks_low              3724           67.81
                     squats_low               565           10.29
             head_checks_medium               222            4.04
               head_checks_high               180            3.28
   railway_join_bar_defects_low               171            3.11
               broken_rail_high               135            2.46
                  squats_medium               112            2.04
                broken_rail_low                84            1.53
  railway_join_bar_defects_high                67            1.22
                    squats_high                67            1.22
railway_join_bar_defects_medium                59            1.07
             broken_rail_medium                48            0.87
                 corrosion_high                25            0.46
               corrosion_medium                18            0.33
          

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=df_classes, x="Annotation Count", y="Defect Class", palette="viridis")
plt.title("RailGuard AI â€” Defect Class Distribution (5,492 Bounding Boxes)", fontsize=14, fontweight="bold")
plt.xlabel("Number of Annotations", fontsize=12)
plt.ylabel("Defect Class", fontsize=12)
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

## 4. Sample Bounding Box Inspector

In [ ]:
def draw_sample(image_path, label_path):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    
    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls_id, cx, cy, bw, bh = map(float, parts)
                    cls_id = int(cls_id)
                    
                    # Convert YOLO normalized coords to pixel coords
                    x1 = int((cx - bw / 2) * w)
                    y1 = int((cy - bh / 2) * h)
                    x2 = int((cx + bw / 2) * w)
                    y2 = int((cy + bh / 2) * h)
                    
                    label_name = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
                    cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (255, 0, 0), 2)
                    cv2.putText(img_rgb, label_name, (x1, max(y1 - 10, 20)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    return img_rgb

# Select 4 sample training images
train_img_dir = os.path.join(DATASET_DIR, "images", "train")
sample_images = glob.glob(os.path.join(train_img_dir, "*.jpg"))[:4]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for idx, img_p in enumerate(sample_images):
    lbl_p = img_p.replace("images\\train", "labels\\train").replace("images/train", "labels/train").replace(".jpg", ".txt")
    img_vis = draw_sample(img_p, lbl_p)
    
    ax = axes[idx // 2, idx % 2]
    ax.imshow(img_vis)
    ax.set_title(os.path.basename(img_p))
    ax.axis("off")

plt.suptitle("RailGuard AI â€” Ground-Truth Annotation Overlay Samples", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()